# Module 2: In-Memory Processing

Run in Google Colab or a local PySpark environment.

In [ ]:
# Run this cell first
!pip install -q pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.appName("LearningModules").getOrCreate()
sc = spark.sparkContext

In [ ]:
# Load datasets (adjust path if needed)
students_df = spark.read.csv("../data/students.csv", header=True, inferSchema=True)
enrollments_df = spark.read.csv("../data/enrollments.csv", header=True, inferSchema=True)

## Caching with cache()

When you call `.cache()` on a DataFrame, Spark stores it in memory after the first action. Subsequent actions on the same DataFrame will be much faster.

In [ ]:
import time

# Without cache - first run
start = time.time()
students_df.filter(col("gpa") > 3.0).count()
print(f"Without cache: {time.time() - start:.4f} seconds")

In [ ]:
# Cache the DataFrame
students_df.cache()

# First action materializes the cache
students_df.count()

# Subsequent actions use the cache
start = time.time()
students_df.filter(col("gpa") > 3.0).count()
print(f"With cache: {time.time() - start:.4f} seconds")

## Persist with Storage Levels

`.persist()` gives you control over where and how data is stored (memory only, disk only, or both).

In [ ]:
from pyspark import StorageLevel

# Persist with memory and disk
enrollments_df.persist(StorageLevel.MEMORY_AND_DISK)

# Trigger materialization
enrollments_df.count()

# Now the DataFrame is persisted
print("Storage level:", enrollments_df.storageLevel)

## Execution Plans with explain()

Use `.explain()` to see how Spark plans to execute a query. This helps you understand optimizations.

In [ ]:
# Show execution plan for a filtered query
high_gpa = students_df.filter(col("gpa") > 3.5)
high_gpa.explain()

In [ ]:
# Extended explain shows more detail
high_gpa.explain(extended=True)

## Timing Comparison

Let's demonstrate the difference caching makes with repeated operations.

In [ ]:
# Unpersist first to reset
students_df.unpersist()

# Time without cache (multiple actions)
start = time.time()
for _ in range(3):
    students_df.groupBy("gpa").count().collect()
no_cache_time = time.time() - start

# Cache and time again
students_df.cache()
students_df.count()  # materialize

start = time.time()
for _ in range(3):
    students_df.groupBy("gpa").count().collect()
cache_time = time.time() - start

print(f"Without cache (3 runs): {no_cache_time:.4f}s")
print(f"With cache (3 runs):    {cache_time:.4f}s")

## Practice Problem 1: Cache and Compare

Cache the enrollments DataFrame and compare the time for running `.count()` before and after caching.

<details><summary>Hint</summary>Unpersist first, time a count(), then cache, materialize with count(), and time another count().</details>

In [ ]:
# Your code here


<details><summary>Solution</summary></details>

In [ ]:
# Solution
enrollments_df.unpersist()

start = time.time()
enrollments_df.count()
print(f"Before cache: {time.time() - start:.4f}s")

enrollments_df.cache()
enrollments_df.count()  # materialize

start = time.time()
enrollments_df.count()
print(f"After cache: {time.time() - start:.4f}s")

## Practice Problem 2: Explain a Join

Join students_df and enrollments_df on student_id, then use `.explain()` to view the execution plan.

<details><summary>Hint</summary>Use .join() with the on parameter, then call .explain() on the result.</details>

In [ ]:
# Your code here


<details><summary>Solution</summary></details>

In [ ]:
# Solution
joined_df = students_df.join(enrollments_df, on="student_id")
joined_df.explain()

## Practice Problem 3: Storage Levels

Persist the joined DataFrame with `DISK_ONLY` storage level and verify the storage level.

<details><summary>Hint</summary>Use .persist(StorageLevel.DISK_ONLY) and then check .storageLevel property.</details>

In [ ]:
# Your code here


<details><summary>Solution</summary></details>

In [ ]:
# Solution
joined_df = students_df.join(enrollments_df, on="student_id")
joined_df.persist(StorageLevel.DISK_ONLY)
joined_df.count()  # materialize
print("Storage level:", joined_df.storageLevel)